# 🏠 Interpretación del Modelo de Viviendas: ¿Qué fija el precio?

En este análisis de **IA Explicable (XAI)**, vamos a auditar el modelo de regresión que estima los precios inmobiliarios. 

El objetivo es entender qué características de una propiedad (habitaciones, metros cuadrados, ubicación) son las que realmente "mueven la aguja" del precio final, permitiendo una valoración transparente y justificada.


In [1]:
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt

# 1. Rutas
path_mod = '../../../models/modelo_viviendas_optimizado_v2.pkl'
path_dat = '../../../data/processed/viviendas_limpio.csv' # Usamos 'limpio'

# 2. Carga
modelo_viviendas = joblib.load(path_mod)
df_viviendas = pd.read_csv(path_dat)

# 3. Preparación de la muestra
cols = modelo_viviendas.feature_names_in_
X_viviendas_all = df_viviendas.reindex(columns=cols, fill_value=0)
n_muestras = min(len(X_viviendas_all), 100) 
X_viviendas = X_viviendas_all.sample(n=n_muestras, random_state=42)

print(f"✅ Viviendas listo. Auditando una muestra de {n_muestras} propiedades.")

c:\Users\txema\Documents\IA_Especialista\ia_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Viviendas listo. Auditando una muestra de 100 propiedades.


## 🌍 Análisis Global: Factores que mueven el Mercado
El **Summary Plot** nos mostrará el impacto de cada variable en el precio de la vivienda. Veremos qué características son "Premium" (suman valor) y cuáles penalizan la tasación.

In [ ]:
# 1. Calculamos valores SHAP globales
explainer_global = shap.TreeExplainer(modelo_viviendas) # type: ignore
shap_values_global = explainer_global.shap_values(X_viviendas)

# 2. Renderizado limpio
plt.close('all') 
shap.summary_plot(shap_values_global, X_viviendas, show=False) # type: ignore
plt.title(f"Viviendas: Factores Globales de Precio (n={n_muestras})", fontsize=14, pad=20)
plt.show()

## 🏘️ Análisis Local: Auditoría de una Propiedad Específica
Aislamos una vivienda concreta para justificar su tasación. El gráfico de **Waterfall** nos permite explicarle a un propietario por qué su casa vale más o menos que el precio medio del barrio ($E[f(X)]$).

In [ ]:
# 1. Objeto de explicación moderno
explainer_local = shap.Explainer(modelo_viviendas, X_viviendas)
exp_viviendas = explainer_local(X_viviendas)

# 2. Selección de la casa (índice 0)
# [Especialista]: En regresión NO usamos [:, :, 1]. Solo el índice del registro.
exp_individual = exp_viviendas[0] 

# 3. Gráfico de Cascada
plt.close('all')
shap.plots.waterfall(exp_individual, max_display=10)